In [1]:
pip install rapidfuzz

  Using cached rapidfuzz-3.12.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
Using cached rapidfuzz-3.12.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pygris


  Using cached pygris-0.1.6-py3-none-any.whl.metadata (2.5 kB)
  Using cached fiona-1.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (56 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached rtree-1.4.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
Using cached pygris-0.1.6-py3-none-any.whl (55 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
Using cached fiona-1.10.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
Using cached rtree-1.4.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (541 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import geopandas as gpd
import folium
from geopy.distance import geodesic
from rapidfuzz import process, fuzz
import matplotlib.pyplot as plt
import numpy as np
from pygris import tracts

In [4]:
stops = pd.read_csv('stops.txt')
stops.head()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url
0,390,10390,19th Avenue & Holloway St,,37.721190,-122.475153,,https://SFMTA.com/10390
1,913,10913,Dublin St & La Grande Ave,,37.719192,-122.425802,,https://SFMTA.com/10913
2,3016,13016,3rd St & 4th St,,37.772618,-122.389786,,https://SFMTA.com/13016
3,3018,13018,Bacon St & San Bruno Ave,,37.727859,-122.402994,,https://SFMTA.com/13018
4,3019,13019,Bacon St & San Bruno Ave,,37.727645,-122.403269,,https://SFMTA.com/13019


In [5]:
routes = pd.read_csv('routes.txt')

In [6]:
routes2=gpd.read_file('Muni_Simple_Routes_20250202.csv')

In [7]:
trips = pd.read_csv('trips.txt')

In [8]:
stoptimes = pd.read_csv('stop_times.txt')

In [9]:
Filteredschools = gpd.read_file('Filtered_Schools.csv')

In [10]:
EnglishScores = pd.read_csv('eladownload2024.csv')

/tmp/ipykernel_138/116707554.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  EnglishScores = pd.read_csv('eladownload2024.csv')


In [11]:
MathScores = pd.read_csv('mathdownload2024 (1).csv')


/tmp/ipykernel_138/4238787621.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  MathScores = pd.read_csv('mathdownload2024 (1).csv')


In [12]:
EnglishScoresschools = EnglishScores.dropna(subset=['countyname'])
EnglishSchools = EnglishScoresschools[EnglishScoresschools['countyname'].str.contains('San Francisco')]
MathScoresschols = MathScores.dropna(subset=['countyname'])
MathSchools = MathScoresschols[MathScoresschols['countyname'].str.contains('San Francisco')]

In [13]:
EnglishHS = EnglishSchools[~EnglishSchools['schoolname'].str.contains('Elementary|Middle', case=False, na=False)]
MathHS = MathSchools[~MathSchools['schoolname'].str.contains('Elementary|Middle', case=False, na=False)]

In [14]:
crimes = pd.read_csv('Police_Department_Incident_Reports__2018_to_Present_20250306.csv')
crimesin2024 = crimes .loc[crimes ['Incident Year']==2024]


In [15]:
publicschools = Filteredschools[Filteredschools['CCSF Entity'].str.contains('SFUSD')]

In [16]:
publichighschools = publicschools[publicschools['Grade Range'].str.contains('9-12')]
publichighschools

,Campus Name,CCSF Entity,Lower Grade,Upper Grade,Grade Range,Category,Map Label,Lower Age,Upper Age,General Type,...,Campus Address,Supervisor District,County FIPS,County Name,Location 1,Neighborhoods (old),Zip Codes,Fire Prevention Districts,Police Districts,Supervisor Districts
11,"Marshall, Thurgood Marshall High School",SFUSD,9,12,9-12,USD Grades 9-12,PS073,14,17,PS,...,"45 CONKLING ST, San Francisco, CA 94124",10,6075,SAN FRANCISCO,"CA\n(37.736309, -122.401649)",1,58,10.0,3.0,8
38,"Hearst, Phoebe Apperson Hearst Home",SFUSD,9,12,9-12,USD Grades 9-12,PS045,14,17,PS,...,"3045 SANTIAGO ST, SAN FRANCISCO 94116",4,6075,SAN FRANCISCO,"CA\n(37.74363, -122.500053)",35,29491,1.0,8.0,3
90,"Burton, Phillip And Sala Burton High School",SFUSD,9,12,9-12,USD Grades 9-12,PS011,14,17,PS,...,"400 MANSELL ST, San Francisco, CA 94134",9,6075,SAN FRANCISCO,"CA\n(37.721546, -122.406555)",28,309,10.0,3.0,7
92,"Washington, George Washington High School",SFUSD,9,12,9-12,USD Grades 9-12,PS121,14,17,PS,...,"600 32ND AVE, San Francisco, CA 94121",1,6075,SAN FRANCISCO,"CA\n(37.777905, -122.491013)",26,55,11.0,6.0,2
135,"Lincoln, Abraham Lincoln High School",SFUSD,9,12,9-12,USD Grades 9-12,PS067,14,17,PS,...,"2162 24TH AVE, San Francisco, CA 94116",4,6075,SAN FRANCISCO,"CA\n(37.746594, -122.48024)",35,29491,1.0,8.0,3
174,Life Learning Academy Charter School,SFUSD,9,12,9-12,USD Charter School,PS064,14,17,PS,...,"651 8TH TI ST, SAN FRANCISCO, CA 94130",6,6075,SAN FRANCISCO,"CA\n(37.825512, -122.367996)",37,62,,2.0,9
194,Gateway High School / Kipp Sf Bay Academy,SFUSD,9,12,9-12,USD Charter School,PS037,14,17,PS,...,"1430 SCOTT ST, San Francisco, CA 94115",5,6075,SAN FRANCISCO,"CA\n(37.783264, -122.436691)",41,29490,13.0,5.0,11
195,Galileo High School,SFUSD,9,12,9-12,USD Grades 9-12,PS035,14,17,PS,...,"1150 FRANCISCO ST, San Francisco, CA 94109",2,6075,SAN FRANCISCO,"CA\n(37.803791, -122.424149)",32,28858,5.0,9.0,1
219,Balboa High School,SFUSD,9,12,9-12,USD Grades 9-12,PS007,14,17,PS,...,"1000 CAYUGA AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.721142, -122.441399)",25,28861,9.0,7.0,6
284,City Arts And Tech High School,SFUSD,9,12,9-12,USD Grades 9-12,PS019,14,17,PS,...,"325 LA GRANDE AVE, San Francisco, CA 94112",11,6075,SAN FRANCISCO,"CA\n(37.718784, -122.424667)",18,309,9.0,7.0,6


In [17]:
public_high_school_names = publichighschools['Campus Name'].tolist()

In [18]:
public_high_school_names

['Marshall, Thurgood Marshall High School',
 'Hearst, Phoebe Apperson Hearst Home',
 'Burton, Phillip And Sala Burton High School',
 'Washington, George Washington High School',
 'Lincoln, Abraham Lincoln High School',
 'Life Learning Academy Charter School',
 'Gateway High School / Kipp Sf Bay Academy',
 'Galileo High School',
 'Balboa High School',
 'City Arts And Tech High School',
 'Independence High School',
 'Wallenberg, Raoul Wallenberg High School',
 'Downtown High School',
 'San Francisco International High School',
 'Jordan, June Jordan High School',
 "O'Connell, John O'Connell High School",
 'Mission High School',
 'Wells, Ida B. Wells High School']

In [19]:
# Function to find similar names with debug
def find_similar_names(name, name_list, threshold=51):
    matches = process.extract(name, name_list, scorer=fuzz.ratio, limit=1)
    if matches:
        print(f"Checking '{name}' against '{matches[0][0]}' with score {matches[0][1]}")
        if matches[0][1] >= threshold:
            return True
    return False

In [20]:
schoolsenglishscores = EnglishHS['schoolname'].apply(lambda x: find_similar_names(x, public_high_school_names))

Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' with score 34.92063492063492
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' w

In [21]:
schoolsmathscores = MathHS['schoolname'].apply(lambda x: find_similar_names(x, public_high_school_names))

Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Court Woodside Learning Ctr' against 'Life Learning Academy Charter School' with score 40.0
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' with score 34.92063492063492
Checking 'S.F. County Opportunity (Hilltop)' against 'City Arts And Tech High School' w

In [22]:
filtered_df_based_on_similar_namesenglish =EnglishHS[schoolsenglishscores]

In [23]:
filtered_df_based_on_similar_namesmath =MathHS[schoolsmathscores]

In [24]:
filtered_df_based_on_similar_namesenglish['schoolname'].unique()

array(['Jordan (June) School for Equity',
       'City Arts & Leadership Academy',
       "Five Keys Independence HS (SF Sheriff's)",
       'S.F. International High', 'San Francisco Public Montessori',
       'Wells (Ida B.) High', 'Downtown High', 'Independence High',
       'Wallenberg (Raoul) Traditional High',
       'Burton (Phillip and Sala) Academic High', 'Balboa High',
       'Marshall (Thurgood) High', 'Life Learning Academy Charter',
       'Gateway High', 'Galileo High', 'Lincoln (Abraham) High',
       'Mission High', "O'Connell (John) High",
       'Washington (George) High', 'San Francisco Community Alternative'],
      dtype=object)

In [25]:
schools_to_remove = [
    'San Francisco Public Montessori',
    'San Francisco Community Alternative'
]

In [26]:
filtered_df_based_on_similar_namesenglish = filtered_df_based_on_similar_namesenglish[
    ~filtered_df_based_on_similar_namesenglish['schoolname'].isin(schools_to_remove)
]

In [27]:
filtered_df_based_on_similar_namesmath = filtered_df_based_on_similar_namesmath[
    ~filtered_df_based_on_similar_namesmath['schoolname'].isin(schools_to_remove)
]

In [28]:
filtered_df_based_on_similar_namesmath = filtered_df_based_on_similar_namesmath[['schoolname', 'studentgroup', 'statuslevel']]
filtered_df_based_on_similar_namesenglish = filtered_df_based_on_similar_namesenglish[['schoolname', 'studentgroup', 'statuslevel']]

In [29]:
mathperformance = filtered_df_based_on_similar_namesmath[filtered_df_based_on_similar_namesmath['studentgroup'].str.contains('ALL')]
englishperformance = filtered_df_based_on_similar_namesenglish[filtered_df_based_on_similar_namesenglish['studentgroup'].str.contains('ALL')]

In [30]:
combined_statuslevel = {}

In [31]:
for index, row in mathperformance.iterrows():
    schoolname = row['schoolname']
    statuslevel = row['statuslevel']
    if schoolname in combined_statuslevel:
        combined_statuslevel[schoolname] += statuslevel
    else:
        combined_statuslevel[schoolname] = statuslevel

In [32]:
for index, row in englishperformance.iterrows():
    schoolname = row['schoolname']
    statuslevel = row['statuslevel']
    if schoolname in combined_statuslevel:
        combined_statuslevel[schoolname] += statuslevel
    else:
        combined_statuslevel[schoolname] = statuslevel

In [33]:
combined_performance = pd.DataFrame(list(combined_statuslevel.items()), columns=['schoolname', 'statuslevel'])


In [34]:
combined_performance.sort_values('statuslevel')

,schoolname,statuslevel
0,Jordan (June) School for Equity,2
2,Five Keys Independence HS (SF Sheriff's),2
3,S.F. International High,2
4,Wells (Ida B.) High,2
6,Independence High,2
5,Downtown High,2
10,Marshall (Thurgood) High,2
11,Life Learning Academy Charter,2
15,Mission High,2
16,O'Connell (John) High,2


In [35]:
combined_performance.to_csv('schoolsranks.csv', index=False)

In [36]:
goodschools = combined_performance[combined_performance['statuslevel'] >= 5]

In [37]:
school_names = [
    "Galileo High School",
    "Burton, Phillip And Sala Burton High School",
    "Gateway High School / Kipp Sf Bay Academy",
    "Washington, George Washington High School",
    "Lincoln, Abraham Lincoln High School"
]

In [38]:
df_schools = pd.DataFrame(school_names, columns=["School Name"])

In [39]:

goodschoollocation = publichighschools[publichighschools['Campus Name'].isin(df_schools['School Name'])]

In [40]:
visiblecrimesin2024 = crimesin2024[crimesin2024['Incident Category'].str.contains(
    'Larceny Theft|Vandalism|Disorderly Conduct|Assault|Weapons Carrying Etc', 
    regex=True, na=False
)]

In [41]:
visiblecrimesin2024 = visiblecrimesin2024.dropna(subset=['Latitude', 'Longitude'])
visiblecrimesin2024.head()

,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Row ID,Incident ID,Incident Number,CAD Number,...,Longitude,Point,Neighborhoods,ESNCAG - Boundary File,Central Market/Tenderloin Boundary Polygon - Updated,Civic Center Harm Reduction Project Boundary,HSOC Zones as of 2018-06-05,Invest In Neighborhoods (IIN) Areas,Current Supervisor Districts,Current Police Districts
1644,2024/09/05 06:15:00 AM,2024/09/05,06:15,2024,Thursday,2024/09/05 07:10:00 AM,142126304134,1421263,240557895,242490594.0,...,-122.409309,POINT (-122.40930938720703 37.78434753417969),20.0,NaN,1.0,1.0,NaN,NaN,10.0,5.0
2273,2024/08/09 12:00:00 AM,2024/08/09,00:00,2024,Friday,2025/01/23 08:46:00 AM,145657206125,1456572,250043931,250230802.0,...,-122.402855,POINT (-122.4028549194336 37.734073638916016),87.0,NaN,NaN,NaN,NaN,NaN,9.0,2.0
2288,2024/05/08 03:09:00 PM,2024/05/08,15:09,2024,Wednesday,2024/05/08 09:28:00 PM,138885406301,1388854,240290863,241292953.0,...,-122.411720,POINT (-122.4117202758789 37.79629135131836),107.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
2296,2024/05/23 08:55:00 AM,2024/05/23,08:55,2024,Thursday,2024/05/23 08:57:00 AM,139278106362,1392781,240322933,241440712.0,...,-122.452568,POINT (-122.45256805419922 37.76451110839844),29.0,NaN,NaN,NaN,NaN,NaN,11.0,7.0
2301,2024/05/16 09:30:00 PM,2024/05/16,21:30,2024,Thursday,2024/05/17 12:05:00 PM,139270506244,1392705,246066763,NaN,...,-122.436111,POINT (-122.43611145019531 37.76082229614258),38.0,NaN,NaN,NaN,5.0,NaN,5.0,3.0


In [42]:
larceny = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Larceny Theft')]

In [43]:
vandalism = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Vandalism')]

In [44]:
robbery = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Robbery')]

In [45]:
disorder = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Disorderly Conduct')]

In [46]:
assualt = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Assault')]

In [47]:
weapon = visiblecrimesin2024[visiblecrimesin2024['Incident Category'].str.contains('Weapons Carrying Etc')]

In [48]:
# Function to extract and clean coordinates from the 'Location 1' field
def extract_coordinates(location):
    try:
        # Assuming the location is a string of format "CA\n(latitude, longitude)"
        parts = location.split('(')[1].strip(')').split(',')
        # Convert to float and return as a tuple
        return float(parts[0]), float(parts[1])
    except (ValueError, IndexError):
        # If conversion fails, return None
        return None

In [49]:
# Function to calculate distance between two coordinates
def calculate_distance(coord1, coord2):
    return geodesic(coord1, coord2).miles

In [50]:
# Filter stops within 0.15 miles of any school
filtered_stops = []

for stop_idx, stop_row in stops.iterrows():
    stop_coord = (stop_row['stop_lat'], stop_row['stop_lon'])
    for school_idx, school_row in goodschoollocation.iterrows():
        # Extract school coordinates
        school_coord = extract_coordinates(school_row['Location 1'])
        if school_coord:
            # Calculate distance
            distance = calculate_distance(stop_coord, school_coord)
            if distance <= 0.15:
                filtered_stops.append(stop_row)
                break

In [51]:
filtered_stops_df = pd.DataFrame(filtered_stops)
filtered_stops_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 32 to 3115
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   stop_id    36 non-null     int64  
 1   stop_code  36 non-null     int64  
 2   stop_name  36 non-null     object 
 3   stop_desc  36 non-null     object 
 4   stop_lat   36 non-null     float64
 5   stop_lon   36 non-null     float64
 6   zone_id    36 non-null     object 
 7   stop_url   36 non-null     object 
dtypes: float64(2), int64(2), object(4)
memory usage: 2.5+ KB


In [52]:
sf_map = folium.Map(location=[37.7749, -122.4194], zoom_start=12)

In [53]:
# Add filtered Muni stops to the map
for idx, row in filtered_stops_df.iterrows():
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=3,  # Radius of the circle marker
        color='blue',  # Border color of the circle marker
        fill=True,  # Fill the circle marker
        fill_color='blue'  # Fill color of the circle marker
    ).add_to(sf_map)

In [54]:
# Add schools to the map with red color
for idx, row in goodschoollocation.iterrows():
    school_coord = extract_coordinates(row['Location 1'])
    if school_coord:
        folium.CircleMarker(
            location=school_coord,
            radius=3,  # Radius of the circle marker
            color='green',  # Border color of the circle marker
            fill=True,  # Fill the circle marker
            fill_color='green',  # Fill color of the circle marker
            popup=row['Campus Name']  # Popup with school name
        ).add_to(sf_map)

In [55]:
filtered_stops_df

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url
32,3055,13055,Balboa St & 30th Ave,,37.776114,-122.489958,,https://SFMTA.com/13055
33,3056,13056,Balboa St & 30th Ave,,37.775989,-122.489912,,https://SFMTA.com/13056
34,3057,13057,Balboa St & 32nd Ave,,37.776024,-122.491838,,https://SFMTA.com/13057
361,3551,13551,33rd Ave & Anza St,,37.777861,-122.493249,,https://SFMTA.com/13551
362,3552,13552,33rd Ave & Anza St,,37.777692,-122.493077,,https://SFMTA.com/13552
649,3957,13957,Chestnut St & Van Ness Ave,,37.802199,-122.424894,,https://SFMTA.com/13957
871,4274,14274,Geary Blvd & 30th Ave,,37.779727,-122.490190,,https://SFMTA.com/14274
872,4275,14275,Geary Blvd & 32nd Ave,,37.779629,-122.491714,,https://SFMTA.com/14275
887,4293,14293,Geary Blvd & Divisadero St,,37.783428,-122.439366,,https://SFMTA.com/14293
995,4421,14421,Divisadero St & Geary Blvd,,37.783375,-122.439412,,https://SFMTA.com/14421


In [56]:
stopslist = filtered_stops_df['stop_id'].tolist()


In [57]:
stoptimes['stop_id'] = stoptimes['stop_id'].astype(str)
stopslist = [str(stop) for stop in stopslist]

stoptimesatbus = stoptimes[stoptimes['stop_id'].isin(stopslist)]

In [58]:
tripids = stoptimesatbus['trip_id'].tolist()

In [59]:
trippier = trips[trips['trip_id'].isin(tripids)]


In [60]:
trippy = trippier['route_id'].tolist()


In [61]:
onlyroutes = routes[routes['route_id'].isin(trippy)]


In [62]:
onlynecessaryroutes = onlyroutes['route_id'].tolist()

In [63]:
routesonmap = routes2[routes2['ROUTE_NAME'].isin(onlynecessaryroutes)]
routesonmap

,objectid,PATTERN,PATTERNID,ROUTE_NAME,DIRECTION,PATTERN_TYPE,SUB_TYPE,PATTERN_VERSION,LINEABBR,SIGNID,SERVICE_CA,shape,data_as_of,data_loaded_at
10,13806,18 I F00,216628,18,I,F,0,0,018,147,Grid,"MULTILINESTRING ((-122.475028 37.725741, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
11,13807,18 O F01,216626,18,O,F,0,1,018,147,Grid,"MULTILINESTRING ((-122.499615 37.785033, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
12,13808,19 I F00,216631,19,I,F,0,0,019,147,Grid,"MULTILINESTRING ((-122.366889 37.728795, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
13,13809,19 O F00,216630,19,O,F,0,0,019,147,Grid,"MULTILINESTRING ((-122.423253 37.806348, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
24,13820,24 I F00,216665,24,I,F,0,0,024,147,Frequent Local,"MULTILINESTRING ((-122.390928 37.734118, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
25,13821,24 O F00,216661,24,O,F,0,0,024,147,Frequent Local,"MULTILINESTRING ((-122.433117 37.792707, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
30,13826,28 I F10,216677,28,I,F,1,0,028,147,Frequent Local,"MULTILINESTRING ((-122.468804 37.70588, -122.4...",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
31,13827,28 O F10,216674,28,O,F,1,0,028,147,Frequent Local,"MULTILINESTRING ((-122.412342 37.807695, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
34,13830,29 I F00,216685,29,I,F,0,0,029,147,Grid,"MULTILINESTRING ((-122.394796 37.722895, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM
35,13831,29 O F00,216681,29,O,F,0,0,029,147,Grid,"MULTILINESTRING ((-122.481575 37.792025, -122....",11/02/2024 08:47:18 PM,11/04/2024 09:42:45 AM


In [64]:
routesonmap['ROUTE_NAME'].unique()

array(['18', '19', '24', '28', '29', '30', '31', '38', '38R', '48', '49',
       '56', '66', '90', '91'], dtype=object)

In [65]:
routesonmap = routesonmap[~routesonmap['ROUTE_NAME'].isin(['90', '91'])]

In [89]:
# Save the filtered data to a new CSV file
routesonmap.to_csv('filtered_routesonmap.csv', index=False)

In [90]:
# Get unique route names
unique_routes = routesonmap['ROUTE_NAME'].unique()

# Create and save a CSV file for each unique route
for route in unique_routes:
    route_df = routesonmap[routesonmap['ROUTE_NAME'] == route]
    filename = f'route_{route}.csv'
    route_df.to_csv(filename, index=False)
    print(f"Created {filename}")

print("All unique route files have been created.")

Created route_18.csv
Created route_19.csv
Created route_24.csv
Created route_28.csv
Created route_29.csv
Created route_30.csv
Created route_31.csv
Created route_38.csv
Created route_38R.csv
Created route_48.csv
Created route_49.csv
Created route_56.csv
Created route_66.csv
All unique route files have been created.


In [66]:
for idx, row in routesonmap.iterrows():
    # Extract the route shape from the 'shape' column
    shape = row['shape'].replace('MULTILINESTRING ((', '').replace('))', '')
    points = shape.split(', ')
    coordinates = [(float(point.split()[1]), float(point.split()[0])) for point in points]
    
    folium.PolyLine(
        coordinates,
        color='red',
        weight=2.5,
        opacity=1
    ).add_to(sf_map)

In [67]:
stops2=gpd.read_file('Muni_Stops_20250126.csv')

In [68]:
stops2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3285 entries, 0 to 3284
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   OBJECTID                      3285 non-null   object
 1   STOPNAME                      3285 non-null   object
 2   TRAPEZESTOPABBR               3285 non-null   object
 3   RUCUSSTOPABBR                 3285 non-null   object
 4   STOPID                        3285 non-null   object
 5   LATITUDE                      3285 non-null   object
 6   LONGITUDE                     3285 non-null   object
 7   ACCESSIBILITYMASK             3285 non-null   object
 8   ATSTREET                      3285 non-null   object
 9   ONSTREET                      3285 non-null   object
 10  POSITION                      3285 non-null   object
 11  ORIENTATION                   3285 non-null   object
 12  SERVICEPLANNINGSTOPTYPE       3285 non-null   object
 13  SHELTER           

In [69]:
stoptimes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 961721 entries, 0 to 961720
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   trip_id              961721 non-null  int64 
 1   arrival_time         961721 non-null  object
 2   departure_time       961721 non-null  object
 3   stop_id              961721 non-null  object
 4   stop_sequence        961721 non-null  int64 
 5   stop_headsign        961721 non-null  object
 6   pickup_type          961721 non-null  object
 7   drop_off_type        961721 non-null  object
 8   shape_dist_traveled  961721 non-null  object
dtypes: int64(2), object(7)
memory usage: 66.0+ MB


In [70]:
trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25533 entries, 0 to 25532
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   route_id               25533 non-null  object
 1   service_id             25533 non-null  int64 
 2   trip_id                25533 non-null  int64 
 3   trip_headsign          25533 non-null  object
 4   direction_id           25533 non-null  int64 
 5   block_id               25533 non-null  int64 
 6   shape_id               25533 non-null  int64 
 7   wheelchair_accessible  25533 non-null  int64 
 8   bikes_allowed          25533 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 1.8+ MB


In [71]:
routesneededforstops = routesonmap['ROUTE_NAME'].tolist()


In [72]:
tripsneededforstops = trips[trips['route_id'].isin(routesneededforstops)]



In [73]:
tripsneededforstops2 =tripsneededforstops['trip_id'].tolist()


In [74]:
stoptimes['trip_id'].unique()

array([11708204, 11708205, 11708206, ..., 11756026, 11756027, 11756028])

In [75]:
stopidsforroutes = stoptimes[stoptimes['trip_id'].isin(tripsneededforstops2)]

In [76]:
stopidsfortoutes2 = stopidsforroutes['stop_id'].tolist()


In [77]:
stopidsforroutes3 = stops2[stops2['STOPID'].isin(stopidsfortoutes2)]

In [78]:
stopidsroute4 = stopidsforroutes3['STOPID'].unique()

In [79]:
stopidsroute5 =stopidsforroutes3[stopidsforroutes3['STOPID'].isin(stopidsroute4)]
stopidsroute5

,OBJECTID,STOPNAME,TRAPEZESTOPABBR,RUCUSSTOPABBR,STOPID,LATITUDE,LONGITUDE,ACCESSIBILITYMASK,ATSTREET,ONSTREET,...,SIGNUPID,SUPERVISOR_DISTRICT,shape,data_as_of,data_loaded_at,Current Police Districts,Current Supervisor Districts,Analysis Neighborhoods,Neighborhoods,SF Find Neighborhoods
2,73681,Mansell St&Somerset St S-NS/PS,MANSSOM0,MANSSOME,5351,37.720417999999995,-122.40509399999999,,SOMERSET ST,MANSELL ST,...,147,9,POINT (-122.405094 37.720418),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,2,9,25,75,75
6,73708,Gilman Ave&3rd St E-NS/PS,GLMN 3S1,GLMN 3ST,4783,37.722457999999996,-122.395415,,3RD ST,GILMAN AVE,...,147,10,POINT (-122.395415 37.722458),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,2,9,1,88,88
8,74008,Mission St&26th St SW-FS/BZ,MISS26S0,MISS26ST,5568,37.74857,-122.41818,,26TH ST,MISSION ST,...,147,9,POINT (-122.41818 37.74857),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,3,2,20,53,53
15,72765,Balboa St&Park Presidio Blvd SW-NS,BBOAPKP1,BBOAPKPR,3071,37.776787999999996,-122.472411,,PARK PRESIDIO BLVD,BALBOA ST,...,147,1,POINT (-122.472411 37.776788),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,8,4,29,5,5
16,75844,Lombard St&Fillmore St SE-FS/BZ,LOMBFIL1,LOMBFILL,5274,37.799817999999995,-122.435878,,WEBSTER ST,LOMBARD ST,...,147,2,POINT (-122.435878 37.799818),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,4,6,13,15,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3269,75283,46th Ave&Noriega St SW-FS/PS,46AVNOR0,46AVNORI,3587,37.752837,-122.505482,,NORIEGA ST,46TH AVE,...,147,4,POINT (-122.505482 37.752837),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,10,7,35,39,39
3275,73804,Galvez Ave&Hill Ave MB-NS/BZ,GLVZHILL,MNSUHUSY,5633,37.728795,-122.36702700000001,,ROBINSON ST,GALVEZ AVE,...,147,10,POINT (-122.367027 37.728795),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,2,9,1,78,78
3280,72901,46th Ave&Vicente St SE-NS/PS,46AVVCT1,46AVVCTE,3603,37.737973,-122.504268,1,VICENTE ST,46TH AVE,...,147,4,POINT (-122.504268 37.737973),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,10,7,35,40,40
3281,75753,Turk St&Stanyan St SE-FS/BZ,TURKSTA1,TURKSTAN,6732,37.777557,-122.455035,,STANYAN ST,TURK BLVD,...,147,1,POINT (-122.455035 37.777557),11/02/2024 08:47:21 PM,11/04/2024 09:42:39 AM,7,4,18,12,12


In [80]:
stopidsroute5.to_csv('stopidsforgoodschoolbus.csv', index=False)

In [81]:
censustracts = pd.read_csv('Census_2020__Tracts_for_San_Francisco_20250310.csv')
censustracts

,the_geom,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,data_loaded_at,data_as_of
0,MULTIPOLYGON (((-122.427223 37.715548999999996...,6,75,980501,6075980501,9805.01,Census Tract 9805.01,G5020,S,1471536,9769,37.716208,-122.419346,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
1,"MULTIPOLYGON (((-122.453206 37.768249, -122.45...",6,75,17102,6075017102,171.02,Census Tract 171.02,G5020,S,294894,0,37.765435,-122.450475,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
2,"MULTIPOLYGON (((-122.414995 37.787454, -122.41...",6,75,12302,6075012302,123.02,Census Tract 123.02,G5020,S,92653,0,37.787022,-122.412097,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
3,"MULTIPOLYGON (((-122.432351 37.773779, -122.43...",6,75,16801,6075016801,168.01,Census Tract 168.01,G5020,S,226534,0,37.771331,-122.429013,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
4,"MULTIPOLYGON (((-122.41246100000001 37.791627,...",6,75,11902,6075011902,119.02,Census Tract 119.02,G5020,S,93053,0,37.790996,-122.409807,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,"MULTIPOLYGON (((-122.426462 37.766272, -122.42...",6,75,20201,6075020201,202.01,Census Tract 202.01,G5020,S,140617,0,37.764788,-122.424095,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
240,MULTIPOLYGON (((-122.393618 37.783077999999996...,6,75,61508,6075061508,615.08,Census Tract 615.08,G5020,S,227943,152297,37.785752,-122.386381,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
241,"MULTIPOLYGON (((-122.430747 37.798648, -122.42...",6,75,13001,6075013001,130.01,Census Tract 130.01,G5020,S,181456,0,37.797663,-122.427228,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM
242,"MULTIPOLYGON (((-122.424835 37.802268, -122.42...",6,75,10201,6075010201,102.01,Census Tract 102.01,G5020,S,186873,0,37.801303,-122.421215,03/04/2022 12:00:00 AM,02/01/2021 12:00:00 AM


In [82]:
from shapely.geometry import Point

In [83]:
geometry = [Point(xy) for xy in zip(stopidsroute5['LONGITUDE'], stopidsroute5['LATITUDE'])]
bus_stops_gdf = gpd.GeoDataFrame(stopidsroute5, geometry=geometry)

In [84]:
visiblecrimesin2024['Incident Time'] = pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.strftime('%I:%M %p')
filtered_crimes_by_time_and_day = visiblecrimesin2024[
    ((pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time >= pd.to_datetime('06:00 AM').time()) &
     (pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time <= pd.to_datetime('08:00 AM').time())) |
    ((pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time >= pd.to_datetime('03:00 PM').time()) &
     (pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time <= pd.to_datetime('06:00 PM').time())) &
    (~visiblecrimesin2024['Incident Day of Week'].isin(['Saturday', 'Sunday']))&
      (pd.to_datetime(visiblecrimesin2024['Report Datetime']).dt.date >= pd.to_datetime('2024-01-01').date()) &
    (pd.to_datetime(visiblecrimesin2024['Report Datetime']).dt.date <= pd.to_datetime('2024-12-31').date())
]

/tmp/ipykernel_138/1364634421.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  visiblecrimesin2024['Incident Time'] = pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.strftime('%I:%M %p')
/tmp/ipykernel_138/1364634421.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ((pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time >= pd.to_datetime('06:00 AM').time()) &
/tmp/ipykernel_138/1364634421.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  (pd.to_datetime(visiblecrimesin2024['Incident Time']).dt.time <= pd.to_datetime('08:00 AM').time())) |
/tmp/ipykern

In [85]:
filtered_crimes_by_time_and_day = filtered_crimes_by_time_and_day[
    pd.to_datetime(filtered_crimes_by_time_and_day['Report Datetime']).dt.year != 2025
]


In [86]:
filtered_crimes_by_time_and_day

,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Row ID,Incident ID,Incident Number,CAD Number,...,Longitude,Point,Neighborhoods,ESNCAG - Boundary File,Central Market/Tenderloin Boundary Polygon - Updated,Civic Center Harm Reduction Project Boundary,HSOC Zones as of 2018-06-05,Invest In Neighborhoods (IIN) Areas,Current Supervisor Districts,Current Police Districts
1644,2024/09/05 06:15:00 AM,2024/09/05,06:15 AM,2024,Thursday,2024/09/05 07:10:00 AM,142126304134,1421263,240557895,242490594.0,...,-122.409309,POINT (-122.40930938720703 37.78434753417969),20.0,NaN,1.0,1.0,NaN,NaN,10.0,5.0
2288,2024/05/08 03:09:00 PM,2024/05/08,03:09 PM,2024,Wednesday,2024/05/08 09:28:00 PM,138885406301,1388854,240290863,241292953.0,...,-122.411720,POINT (-122.4117202758789 37.79629135131836),107.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
2306,2024/05/07 04:15:00 PM,2024/05/07,04:15 PM,2024,Tuesday,2024/05/07 08:39:00 PM,138857806314,1388578,240288476,241282340.0,...,-122.407692,POINT (-122.4076919555664 37.7801628112793),32.0,NaN,1.0,1.0,1.0,NaN,10.0,1.0
2320,2024/05/07 05:05:00 PM,2024/05/07,05:05 PM,2024,Tuesday,2024/05/08 04:10:00 PM,138876806373,1388768,240290108,241292157.0,...,-122.372658,POINT (-122.3726577758789 37.824119567871094),36.0,NaN,NaN,NaN,NaN,NaN,10.0,1.0
2370,2024/05/08 05:25:00 PM,2024/05/08,05:25 PM,2024,Wednesday,2024/05/08 05:25:00 PM,138879512080,1388795,240290465,241292434.0,...,-122.498360,POINT (-122.49835968017578 37.71369552612305),43.0,NaN,NaN,NaN,NaN,NaN,8.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
927500,2024/12/30 05:00:00 PM,2024/12/30,05:00 PM,2024,Monday,2024/12/31 02:30:00 PM,145747906244,1457479,246169204,NaN,...,-122.408401,POINT (-122.40840148925781 37.788291931152344),19.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
927505,2024/12/30 05:23:00 PM,2024/12/30,05:23 PM,2024,Monday,2024/12/30 09:18:00 PM,145749706374,1457497,246169248,NaN,...,-122.405663,POINT (-122.4056625366211 37.80667495727539),99.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
927838,2024/12/26 05:45:00 PM,2024/12/26,05:45 PM,2024,Thursday,2024/12/30 02:49:00 PM,145749506224,1457495,246169191,NaN,...,-122.396584,POINT (-122.3965835571289 37.79458999633789),108.0,NaN,NaN,NaN,NaN,NaN,3.0,6.0
932898,2024/12/18 05:04:00 PM,2024/12/18,05:04 PM,2024,Wednesday,2024/12/18 09:11:00 PM,146280506372,1462805,246169408,NaN,...,-122.405609,POINT (-122.40560913085938 37.716590881347656),75.0,NaN,NaN,NaN,NaN,NaN,9.0,9.0


In [87]:
# Function to extract coordinates from a DataFrame row
def extract_coordinates2(row):
    try:
        # Extract latitude and longitude from the row
        latitude = float(row['LATITUDE'])
        longitude = float(row['LONGITUDE'])
        return latitude, longitude
    except (ValueError, KeyError):
        # If conversion or key access fails, return None
        return None

In [88]:

# Convert the DataFrame to a CSV file
filtered_crimes_df.to_csv('filtered_crimes.csv', index=False)

NameError: name 'filtered_crimes_df' is not defined

In [ ]:
crimesbystops = pd.read_csv('filtered_crimes.csv')

In [ ]:

    for _, crime in crimesbystops.iterrows():
        folium.CircleMarker(
            location=[crime["Latitude"], crime["Longitude"]],
            color="purple",
            fill=True,
            fill_color="purple",
            radius=.00001
        ).add_to(sf_map)

In [ ]:
sf_map

In [ ]:
# Filter stops within 0.01 miles of any stop
filtered_crimes = []

for crime_idx, crime_row in filtered_crimes_by_time_and_day.iterrows():
    crime_coord = (crime_row['Latitude'], crime_row['Longitude'])
    for stop_idx, stop_row in stopidsforroutes3.iterrows():
        # Extract stop coordinates
        stop_coord = extract_coordinates2(stop_row)
        if stop_coord:
            # Calculate distance
            distance = calculate_distance(crime_coord, stop_coord)
            if distance <= 0.01:  # 0.01 miles
                filtered_crimes.append(crime_row)
                break


In [ ]:
filtered_crimes_df = pd.DataFrame(filtered_crimes)